## Estructura Inicial para la Descarga de los Archivos de rtve

In [1]:
import time
import json
import os
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

In [ ]:
# ══════════════════════════════════════════════════════
# CONFIGURACIÓN
# ══════════════════════════════════════════════════════
URL_BUSCADOR = "https://23fbuscador.rtve.es/" # Pagina desde donde se descaran los documentos
OUTPUT_DIR   = "documentos_23f"
DELAY        = 2
HEADLESS     = True   # Se cambia a False si quiero ver el navegador "haciendo el trabajo"
# ══════════════════════════════════════════════════════

def crear_driver():
    options = Options()
    if HEADLESS:
        options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    return webdriver.Chrome(options=options)

# ══════════════════════════════════════════════════════
# CAMBIAR EL PAGINADO A 200 QUE ES EL MAXIMO QUE PERMITE LA PAGINA 
# ══════════════════════════════════════════════════════

def seleccionar_200_por_pagina(driver):
    """
    Selecciona la opción '200 / página' para cargar todos los documentos
    en una sola página y no tener que paginar.
    """
    try:
        wait = WebDriverWait(driver, 15)
        # Primer interto con CSS
        selectores_css = [
            "select[name='page-size-select']",
            "select.page-size",
            "select.pagination-size",
        ]
        select_elem = None
        for sel in selectores_css:
            try:
                select_elem = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, sel)))
                break
            except TimeoutException:
                continue
        # Segundo intenfo por XPath si no funciona por CSS
        if select_elem is None:
            select_elem = wait.until(EC.presence_of_element_located(
                (By.XPATH, "//select[option[contains(text(),'200')]]")
            ))

        select = Select(select_elem)
        # Guardar texto previo a hacer clic para evitar StaleElementReferenceException
        texto_opcion = next((o.text for o in select.options if "200" in o.text), None)
        if texto_opcion:
            select.select_by_visible_text(texto_opcion)
            print(f"✔ Seleccionada opción: '{texto_opcion}'")
            time.sleep(3)
            return True

    except (TimeoutException, NoSuchElementException) as e:
        print(f"⚠ No se pudo seleccionar 200/página: {e}")

    return False

# ══════════════════════════════════════════════════════
# EXTRAER LA TABLA CON EL LISTADO DE LOS DOCUMENTOS DISPONIBLES
# ══════════════════════════════════════════════════════

def extraer_tabla(driver):
    """
    Extrae todas las filas de la tabla de documentos.
    """
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, "//table//tbody//tr"))
        )
    except TimeoutException:
        print("⚠ No apareció la tabla")
        return []

    filas = driver.find_elements(By.XPATH, "//table//tbody//tr")
    print(f"  → {len(filas)} filas en la tabla")

    # Cabeceras
    try:
        ths = driver.find_elements(By.XPATH, "//table//thead//th")
        cabeceras = [th.text.strip() for th in ths]
        print(f"  ℹ Columnas: {cabeceras}")
    except Exception:
        cabeceras = []

    documentos = []
    for i, fila in enumerate(filas):
        celdas = fila.find_elements(By.TAG_NAME, "td")
        doc = {"fila_num": i + 1}

        for j, celda in enumerate(celdas):
            col_name = cabeceras[j] if j < len(cabeceras) else f"col_{j+1}"
            col_name = col_name.lower().replace(" ", "_") or f"col_{j+1}"

            try:
                a = celda.find_element(By.TAG_NAME, "a")
                doc["url"]    = a.get_attribute("href")
                doc[col_name] = a.text.strip()
            except NoSuchElementException:
                doc[col_name] = celda.text.strip()

        documentos.append(doc)

    return documentos


def extraer_transcripcion(driver, url):

    driver.get(url)
    time.sleep(DELAY)

    # XPath confirmado a traves de la revision de la pagina https://23fbuscador.rtve.es/
    # uso del // para generalizar
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//main//section[3]//article//pre"))
        )
        elementos = driver.find_elements(By.XPATH, "//main//section[3]//article//pre")
        textos = [e.text.strip() for e in elementos if e.text.strip()]
        if textos:
            return "\n\n".join(textos)
    except TimeoutException:
        pass

    # Fallback 1: cualquier <pre> en main
    try:
        elementos = driver.find_elements(By.XPATH, "//main//pre")
        textos = [e.text.strip() for e in elementos if e.text.strip()]
        if textos:
            print("    ⚠ fallback: //main//pre")
            return "\n\n".join(textos)
    except Exception:
        pass

    # Fallback 2: todo el <main>
    try:
        print("    ⚠ fallback: texto completo de <main>")
        return driver.find_element(By.TAG_NAME, "main").text.strip()
    except Exception:
        return ""

def nombre_seguro(texto, fallback):
    nombre = re.sub(r'[\\/*?:"<>|]', "", texto or fallback)
    nombre = nombre.replace(" ", "_").strip("._")
    return nombre[:80] or fallback

def guardar(doc, transcripcion):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    titulo = next(
        (v for k, v in doc.items() if k not in ("fila_num", "url") and v),
        f"doc_{doc['fila_num']}"
    )
    base = nombre_seguro(titulo, f"doc_{doc['fila_num']}")

    # .txt legible
# ════════════════════════════════════════════════════════════════════════════════════════════════════════════
# Se deja comentado la inclusion del URL, titulo, numero de paginas, resumen y palabras claves en el archivo TXT
# ════════════════════════════════════════════════════════════════════════════════════════════════════════════
    with open(os.path.join(OUTPUT_DIR, f"{base}.txt"), "w", encoding="utf-8") as f:
#        for k, v in doc.items():
#            if k != "fila_num" and v:
#                f.write(f"{k.upper():15}: {v}\n")
#        f.write("\n" + "═" * 60 + "\n\n")
        f.write(transcripcion)
# ══════════════════════════════════════════════════════
# Se incluye la generacion del json para revision posterior
# ══════════════════════════════════════════════════════
    # .json con todo
    with open(os.path.join(OUTPUT_DIR, f"{base}.json"), "w", encoding="utf-8") as f:
        json.dump({**doc, "transcripcion": transcripcion}, f, ensure_ascii=False, indent=2)

    return base, len(transcripcion)

In [3]:
# ══════════════════════════════════════════════════════
# CREACION PARA LA EJECUCION
# ══════════════════════════════════════════════════════

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    driver = crear_driver()
    todos  = []

    print(f"\n{'═'*55}")
    print(f"  Scraper 23F - Documentos desclasificados RTVE")
    print(f"{'═'*55}\n")

    print("🔍 Abriendo buscador en 23fbuscador.rtve.es ...")
    driver.get(URL_BUSCADOR)
    time.sleep(4)

    print("⚙  Seleccionando 200 documentos por página...")
    seleccionar_200_por_pagina(driver)

    print("\n📋 Extrayendo tabla de documentos...")
    documentos = extraer_tabla(driver)

    if not documentos:
        print("✘ No se encontraron documentos. Prueba con HEADLESS=False para ver qué ocurre.")
        driver.quit()
        return

    print(f"\n🚀 Descargando transcripciones de {len(documentos)} documentos...\n")

    for doc in documentos:
        url = doc.get("url", "")
        titulo = next(
            (v for k, v in doc.items() if k not in ("fila_num", "url") and v),
            f"doc_{doc['fila_num']}"
        )
        titulo_corto = titulo[:55] + ("..." if len(titulo) > 55 else "")
        print(f"  [{doc['fila_num']:>3}/{len(documentos)}] {titulo_corto}")

        if not url:
            print("         ⚠ Sin URL, saltando")
            todos.append(doc)
            continue

        transcripcion = extraer_transcripcion(driver, url)
        base, chars = guardar(doc, transcripcion)
        doc["archivo"] = f"{base}.txt"
        todos.append(doc)
        print(f"         ✔ {base}.txt ({chars:,} caracteres)")

        # Volver al buscador para la siguiente iteración
        driver.get(URL_BUSCADOR)
        time.sleep(DELAY)

    # Índice global
    indice_path = os.path.join(OUTPUT_DIR, "_indice.json")
    with open(indice_path, "w", encoding="utf-8") as f:
        json.dump(todos, f, ensure_ascii=False, indent=2)

    print(f"\n{'═'*55}")
    print(f"  📊 Total documentos : {len(todos)}")
    print(f"  📁 Carpeta          : ./{OUTPUT_DIR}/")
    print(f"  📋 Índice           : {indice_path}")
    print(f"{'═'*55}\n")

    driver.quit()


if __name__ == "__main__":
    main()



═══════════════════════════════════════════════════════
  Scraper 23F - Documentos desclasificados RTVE
═══════════════════════════════════════════════════════

🔍 Abriendo buscador en 23fbuscador.rtve.es ...
⚙  Seleccionando 200 documentos por página...
✔ Seleccionada opción: '200 / página'

📋 Extrayendo tabla de documentos...
  → 167 filas en la tabla
  ℹ Columnas: ['Título', 'Páginas', '', 'Resumen', 'Palabras clave']

🚀 Descargando transcripciones de 167 documentos...

  [  1/167] Vista oral 2/81 del Consejo Supremo de Justicia Militar...
         ✔ Vista_oral_281_del_Consejo_Supremo_de_Justicia_Militar_(20_de_febrero_de_1982).txt (3,934 caracteres)
  [  2/167] Vista oral 2/81 del Consejo Supremo de Justicia Militar...
         ✔ Vista_oral_281_del_Consejo_Supremo_de_Justicia_Militar_(22_de_febrero_de_1982).txt (6,417 caracteres)
  [  3/167] Vista oral 2/81 del Consejo Supremo de Justicia Militar...
         ✔ Vista_oral_281_del_Consejo_Supremo_de_Justicia_Militar_(24_de_febrero_de